In [ ]:
# Runs to analyse: $NESYSY_RUNS (set by run.py), else output/full, else output/subset.
# Figures and tables are written to <runs>/paper_results/.
import os
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
RUNS = Path(os.environ.get("NESYSY_RUNS") or next(
    (ROOT / "output" / m for m in ("full", "subset") if (ROOT / "output" / m).exists()),
    ROOT / "output" / "subset",
))
# the paper shows only points with all 25 runs; a subset has fewer runs per point
MIN_RUNS = int(os.environ.get("NESYSY_MIN_RUNS", 25 if "full" in RUNS.parts else 1))
OUT = RUNS / "paper_results"
OUT.mkdir(parents=True, exist_ok=True)
DBS = sorted(RUNS.glob("**/mlflow.db"))


def search_runs(experiment):
    """mlflow.search_runs of an experiment, over all databases below RUNS."""
    frames = []
    for db in DBS:
        mlflow.set_tracking_uri(f"sqlite:///{db}")
        exp = mlflow.get_experiment_by_name(experiment)
        if exp is not None:
            frames.append(mlflow.search_runs(experiment_ids=[exp.experiment_id]))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


print(f"runs: {RUNS} ({len(DBS)} database(s)), at least {MIN_RUNS} run(s) per point")


In [ ]:
def load_runs(preset_name):
    runs = search_runs(f"sup_train_{preset_name}")
    if runs.empty:
        return runs
    # Drop failed runs (no metrics logged)
    if "metrics.constraint_violation_ratio" in runs.columns:
        runs = runs[runs["metrics.constraint_violation_ratio"].notna()]
    elif "metrics.accuracy" in runs.columns:
        runs = runs[runs["metrics.accuracy"].notna()]
    return runs


def summarize_runs(df, x_col, metric_cols, min_seeds=MIN_RUNS):
    keep_cols = [x_col, *metric_cols]
    tmp = df[[col for col in keep_cols if col in df.columns]].copy()
    tmp[x_col] = pd.to_numeric(tmp[x_col], errors="coerce")
    for col in metric_cols:
        if col in tmp.columns:
            tmp[col] = pd.to_numeric(tmp[col], errors="coerce")
    grp = tmp.dropna(subset=[x_col]).groupby(x_col, as_index=False)
    mean_df = grp.mean(numeric_only=True)
    std_df = grp.std(numeric_only=True)
    count_df = grp.count()
    # Only keep groups with enough seeds
    enough = count_df[metric_cols[0]] >= min_seeds
    mean_df = mean_df[enough.values].reset_index(drop=True)
    std_df = std_df[enough.values].reset_index(drop=True)
    count_df = count_df[enough.values].reset_index(drop=True)
    for col in metric_cols:
        if col in mean_df.columns:
            se = std_df[col] / count_df[col].pow(0.5)
            mean_df[col + "_se"] = se.values
    return mean_df


def plot_with_se(ax, x, y, se, marker, color, label):
    line, = ax.plot(x, y, marker=marker, color=color, label=label)
    ax.fill_between(x, y - se, y + se, alpha=0.2, color=color)
    return line


def se_patch(color):
    return mpatches.Patch(color=color, alpha=0.2, label="\u00b11 SE")


PRESETS = {
    # Fig. 3: vary cvr, output-only constraints
    "cnf_cvr": {"kind": "cnf", "x_col": "params.cvr", "x_label": "Clause-to-variable ratio (cvr)", "file": "fig3a"},
    "bi_cvr": {"kind": "boundary", "x_col": "params.cvr", "x_label": "Clause-to-variable ratio (CVR)", "file": "fig3b"},
    "li_cvr": {"kind": "linear_inequality", "x_col": "params.cvr", "x_label": "Clause-to-variable ratio (CVR)", "file": "fig3c"},
    # Fig. 8, second row: vary cvr, input-output constraints (the first row is Fig. 3)
    "cnf_io_cvr": {"kind": "cnf", "x_col": "params.cvr", "x_label": "Clause-to-variable ratio (cvr)", "file": "fig8d"},
    "bi_io_cvr": {"kind": "boundary", "x_col": "params.cvr", "x_label": "Clause-to-variable ratio", "file": "fig8e"},
    "li_io_cvr": {"kind": "linear_inequality", "x_col": "params.cvr", "x_label": "Clause-to-variable ratio", "file": "fig8f"},
    # Fig. 4: vary the number of regions
    "cnf_regions": {"kind": "cnf", "x_col": "params.n_constraints", "x_label": "Number of constraints", "file": "fig4a"},
    "bi_regions": {"kind": "boundary", "x_col": "params.n_constraints", "x_label": "Number of constraints", "file": "fig4b"},
    "li_regions": {"kind": "linear_inequality", "x_col": "params.n_constraints", "x_label": "Number of constraints", "file": "fig4c"},
    # Fig. 10: vary the number of terms per inequality
    "li_n_terms": {"kind": "linear_inequality", "x_col": "params.constraint.n_terms_min", "x_label": "Number of terms per inequality", "file": "fig10"},
}

REAL_METRICS = [
    "metrics.mean_absolute_error",
    "metrics.mean_consistent_distance",
]
CNF_METRICS = [
    "metrics.constraint_violation_ratio",
    "metrics.accuracy",
]


In [ ]:
# Shared y-limits per (kind, metric) for the cvr presets of Fig. 3 and Fig. 8
cvr_presets = [k for k in PRESETS if "cvr" in k]
scale_dict = {}  # (kind, metric_col) -> (min, max)

for name in cvr_presets:
    info = PRESETS[name]
    runs = load_runs(name)
    if runs.empty:
        continue
    kind = info["kind"]
    is_cnf = kind == "cnf"
    metric_cols = CNF_METRICS if is_cnf else REAL_METRICS
    res = summarize_runs(runs, info["x_col"], metric_cols)
    if res.empty:
        continue
    for col in metric_cols:
        if col in res.columns:
            key = (kind, col)
            lo, hi = res[col].min(), res[col].max()
            if key not in scale_dict:
                scale_dict[key] = (lo, hi)
            else:
                scale_dict[key] = (min(scale_dict[key][0], lo), max(scale_dict[key][1], hi))


In [ ]:
# Figs. 3, 4, 8 (second row) and 10
multi_region_presets = [k for k in PRESETS if "regions" in k]

for preset_name, info in PRESETS.items():
    runs = load_runs(preset_name)
    if runs.empty:
        print(f"No runs found for {preset_name}, skipping.")
        continue

    kind = info["kind"]
    x_col = info["x_col"]
    x_label = info["x_label"]
    is_cnf = kind == "cnf"
    metric_cols = CNF_METRICS if is_cnf else REAL_METRICS

    res = summarize_runs(runs, x_col, metric_cols)
    if res.empty:
        print(f"No data after summarizing {preset_name}, skipping.")
        continue

    fig, ax1 = plt.subplots(figsize=(8, 6))
    ax2 = ax1.twinx()
    x = res[x_col]

    if is_cnf:
        left_m = "metrics.constraint_violation_ratio"
        right_m = "metrics.accuracy"
        line1 = plot_with_se(ax1, x, res[left_m], res[left_m + "_se"], "o", "#1f77b4", "VR")
        line2 = plot_with_se(ax2, x, res[right_m], res[right_m + "_se"], "s", "#d62728", "ACC")
        ax1.set_ylabel("VR", color="#1f77b4", fontsize=20)
        ax2.set_ylabel("ACC (accuracy)", color="#d62728", fontsize=20)
    else:
        left_m = "metrics.mean_consistent_distance"
        right_m = "metrics.mean_absolute_error"
        line1 = plot_with_se(ax1, x, res[left_m], res[left_m + "_se"], "o", "#1f77b4", "MCD")
        line2 = plot_with_se(ax2, x, res[right_m], res[right_m + "_se"], "s", "#d62728", "MAE")
        ax1.set_ylabel("MCD", color="#1f77b4", fontsize=20)
        ax2.set_ylabel("MAE", color="#d62728", fontsize=20)

    # Shared y-limits for CVR presets (per kind)
    if preset_name in cvr_presets and (kind, left_m) in scale_dict:
        lo_l, hi_l = scale_dict[(kind, left_m)]
        lo_r, hi_r = scale_dict[(kind, right_m)]
        ax1.set_ylim(0.9 * lo_l, hi_l * 1.1)
        ax2.set_ylim(0.9 * lo_r, hi_r * 1.1)
        ax1.set_xlim(0.5, 5)

    # Shared x-axis for multi-region presets
    if preset_name in multi_region_presets:
        ax1.set_xlim(1, 5)
        ax1.set_xticks([1, 2, 3, 4, 5])

    ax1.tick_params(axis="y", colors="#1f77b4", labelsize=20)
    ax2.tick_params(axis="y", colors="#d62728", labelsize=20)
    ax1.tick_params(axis="x", labelsize=20)
    ax1.set_xlabel(x_label, fontsize=20)
    ax1.legend(
        [line1, se_patch("#1f77b4"), line2, se_patch("#d62728")],
        [line1.get_label(), "\u00b11 SE", line2.get_label(), "\u00b11 SE"],
        loc="upper center", bbox_to_anchor=(0.5, 1.2), ncol=4, fontsize=20
    )

    plt.tight_layout()
    fig.savefig(OUT / f"{info['file']}.pdf", format="pdf")
    plt.show()


In [ ]:
# Fig. 9: unique outputs and fix distance of the Fig. 3a datasets
import ast

def parse_fix_distance_mean(value):
    """Parse fix_distance_mean from MLflow string like '[(0, 0.1234), ...]'.
    Returns the average across regions."""
    if pd.isna(value):
        return np.nan
    try:
        rows = ast.literal_eval(value)
        distances = [float(d) for _, d in rows]
        return np.mean(distances) if distances else np.nan
    except (ValueError, SyntaxError):
        return np.nan

for preset_name in ["cnf_cvr"]:
    runs = load_runs(preset_name)
    if runs.empty:
        continue

    x_col = PRESETS[preset_name]["x_col"]
    x_label = PRESETS[preset_name]["x_label"]
    fix_col = "params.dataset_info.fix_distance_mean"
    unique_col = "params.dataset_info.n_unique_outputs"

    if fix_col not in runs.columns and unique_col not in runs.columns:
        print(f"No fix_distance_mean or n_unique_outputs for {preset_name}, skipping.")
        continue

    tmp = runs[[x_col]].copy()
    tmp[x_col] = pd.to_numeric(tmp[x_col], errors="coerce")

    has_fix = fix_col in runs.columns
    has_unique = unique_col in runs.columns

    if has_fix:
        tmp["fix_distance"] = runs[fix_col].apply(parse_fix_distance_mean)
    if has_unique:
        tmp["n_unique"] = pd.to_numeric(runs[unique_col], errors="coerce")

    tmp = tmp.dropna(subset=[x_col])
    grp = tmp.groupby(x_col, as_index=False)
    mean_df = grp.mean(numeric_only=True)
    std_df = grp.std(numeric_only=True)
    count_df = grp.count()

    fig, ax1 = plt.subplots(figsize=(10, 6))
    x = mean_df[x_col]

    if has_fix:
        se_fix = std_df["fix_distance"] / count_df["fix_distance"].pow(0.5)
        plot_with_se(ax1, x, mean_df["fix_distance"], se_fix.values, "o", "#1f77b4", "Mean fix distance")
        ax1.set_ylabel("Mean fix distance (Hamming)", color="#1f77b4", fontsize=20)
        ax1.tick_params(axis="y", colors="#1f77b4", labelsize=20)

    if has_unique:
        ax2 = ax1.twinx()
        se_unique = std_df["n_unique"] / count_df["n_unique"].pow(0.5)
        plot_with_se(ax2, x, mean_df["n_unique"], se_unique.values, "s", "#d62728", "Unique outputs")
        ax2.set_ylabel("Number of unique outputs", color="#d62728", fontsize=20)
        ax2.tick_params(axis="y", colors="#d62728", labelsize=20)

    ax1.set_xlabel(x_label, fontsize=20)
    ax1.tick_params(axis="x", labelsize=20)

    lines1 = [se_patch("#1f77b4")] if has_fix else []
    lines2 = [se_patch("#d62728")] if has_unique else []
    handles = ax1.get_legend_handles_labels()[0]
    if has_unique:
        handles += ax2.get_legend_handles_labels()[0]
    labels = [h.get_label() for h in handles]
    handles += lines1 + lines2
    labels += (["\u00b11 SE"] if has_fix else []) + (["\u00b11 SE"] if has_unique else [])
    ax1.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=4, fontsize=20, columnspacing=0.5)

    plt.tight_layout()
    fig.savefig(OUT / "fig9.pdf", format="pdf", bbox_inches="tight")
    plt.show()
